# 05 — CLV Model Explainability with SHAP

## Purpose

This notebook explains the XGBoost future-revenue model created in:

```text
notebooks/04_customer_lifetime_value_prediction.ipynb
        ↓
models/clv_xgboost.joblib
```

The goal is to answer:

- Which features drive model predictions overall?
- Why is a particular customer's predicted future value high or low?
- Which historical behaviors increase predicted value?
- Which behaviors reduce predicted value?

We use **SHAP (SHapley Additive exPlanations)** for both global and customer-level interpretation.

## Explainability workflow

```text
Saved CLV model
      ↓
Historical customer features
      ↓
XGBoost predictions
      ↓
SHAP values
      ├── Global importance
      ├── Beeswarm summary
      ├── Feature dependence
      └── Individual customer explanation
```

### Important

SHAP explains the model's learned associations. It does **not** prove that a feature causally changes customer value.

In [ ]:
from pathlib import Path
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import shap

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MODEL_PATH = PROJECT_ROOT / "models" / "clv_xgboost.joblib"
PREDICTION_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "customer_clv_predictions.csv"
)

SHAP_OUTPUT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "customer_shap_values.csv"
)

print(f"Model: {MODEL_PATH}")
print(f"Prediction data: {PREDICTION_PATH}")

## 1. Load the saved model

In [ ]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Missing {MODEL_PATH}. "
        "Run notebook 04 first."
    )

artifact = joblib.load(MODEL_PATH)

model = artifact["model"]
feature_columns = artifact["features"]

print("Saved features:")
for feature in feature_columns:
    print(" -", feature)

print("\nModel loaded successfully.")

## 2. Load the modeling dataset

The prediction file contains the historical feature matrix used by the model.

We use the same feature order saved inside the model artifact.

In [ ]:
if not PREDICTION_PATH.exists():
    raise FileNotFoundError(
        f"Missing {PREDICTION_PATH}. "
        "Run notebook 04 first."
    )

df = pd.read_csv(PREDICTION_PATH)

missing = [
    column
    for column in feature_columns
    if column not in df.columns
]

if missing:
    raise ValueError(
        f"Missing model features: {missing}"
    )

X = df[feature_columns].copy()

X = X.replace(
    [np.inf, -np.inf],
    np.nan,
)

X = X.fillna(
    X.median(numeric_only=True)
)

print("Feature matrix:", X.shape)
display(X.head())

## 3. Create the SHAP explainer

The saved estimator is a `TransformedTargetRegressor`.

Its underlying estimator is the trained XGBoost regressor.

SHAP's TreeExplainer is applied to that XGBoost model.

In [ ]:
xgb_model = model.regressor_

explainer = shap.TreeExplainer(xgb_model)

# The XGBoost model predicts log(1 + future revenue).
# Therefore SHAP values explain the model in log-target space.
shap_values = explainer.shap_values(X)

shap_values = np.asarray(shap_values)

print("SHAP matrix shape:", shap_values.shape)
print("Feature matrix shape:", X.shape)

## 4. Global SHAP importance

Mean absolute SHAP value measures how strongly a feature contributes to the model's predictions on average.

Higher values indicate greater influence on the model output.

In [ ]:
global_importance = pd.DataFrame({
    "feature": feature_columns,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0),
})

global_importance = global_importance.sort_values(
    "mean_abs_shap",
    ascending=False,
).reset_index(drop=True)

display(global_importance)

fig = px.bar(
    global_importance.sort_values("mean_abs_shap"),
    x="mean_abs_shap",
    y="feature",
    orientation="h",
    title="Global SHAP Feature Importance",
    labels={
        "mean_abs_shap": "Mean |SHAP value|",
        "feature": "Feature",
    },
)
fig.show()

## 5. SHAP summary statistics

For each feature we calculate:

- mean absolute SHAP magnitude,
- average signed SHAP value,
- minimum contribution,
- maximum contribution.

The signed mean helps indicate the overall direction, while individual SHAP values provide the actual customer-specific contribution.

In [ ]:
summary = pd.DataFrame({
    "feature": feature_columns,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0),
    "mean_shap": shap_values.mean(axis=0),
    "min_shap": shap_values.min(axis=0),
    "max_shap": shap_values.max(axis=0),
}).sort_values(
    "mean_abs_shap",
    ascending=False,
)

display(summary.round(4))

## 6. SHAP beeswarm plot

The beeswarm view shows:

- each point = one customer,
- horizontal position = SHAP contribution,
- vertical position = feature,
- color = feature value.

This is one of the most useful plots for explaining nonlinear tree-based models.

In [ ]:
shap.summary_plot(
    shap_values,
    X,
    feature_names=feature_columns,
    show=False,
)

import matplotlib.pyplot as plt

plt.tight_layout()
plt.show()

## 7. SHAP dependence analysis

We inspect the most important features individually.

The plots show how the feature's value relates to its contribution to the model output.

This is useful for discovering nonlinear behavior such as:

```text
low feature value → negative contribution
middle range      → moderate contribution
high value        → strong positive contribution
```

The relationship should be interpreted as a model behavior pattern, not causality.

In [ ]:
top_features = global_importance["feature"].head(
    min(5, len(global_importance))
).tolist()

for feature in top_features:
    shap.dependence_plot(
        feature,
        shap_values,
        X,
        show=False,
    )
    plt.title(f"SHAP Dependence: {feature}")
    plt.tight_layout()
    plt.show()

## 8. Build customer-level SHAP table

Each row corresponds to one customer.

For every feature we store the SHAP contribution in model output space.

Positive value:

> pushes predicted future revenue upward.

Negative value:

> pushes predicted future revenue downward.

In [ ]:
shap_df = pd.DataFrame(
    shap_values,
    columns=[
        f"shap_{feature}"
        for feature in feature_columns
    ],
)

shap_df.insert(
    0,
    "customer_unique_id",
    df["customer_unique_id"].values,
)

if "predicted_future_revenue" in df.columns:
    shap_df["predicted_future_revenue"] = (
        df["predicted_future_revenue"].values
    )

if "future_revenue" in df.columns:
    shap_df["future_revenue"] = (
        df["future_revenue"].values
    )

display(shap_df.head())

## 9. Customer-level explanation function

This helper returns the strongest positive and negative model contributions for one customer.

This will later be reused by the FastAPI explainability endpoint.

In [ ]:
def explain_customer(
    customer_unique_id: str,
    top_n: int = 5,
) -> dict:
    """Return the strongest SHAP contributions for one customer."""

    matches = df.index[
        df["customer_unique_id"].astype(str)
        == str(customer_unique_id)
    ].tolist()

    if not matches:
        raise ValueError(
            f"Customer {customer_unique_id} not found."
        )

    row_index = matches[0]

    contributions = pd.DataFrame({
        "feature": feature_columns,
        "feature_value": X.iloc[row_index].values,
        "shap_value": shap_values[row_index],
    })

    positive = (
        contributions
        .sort_values("shap_value", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    negative = (
        contributions
        .sort_values("shap_value", ascending=True)
        .head(top_n)
        .reset_index(drop=True)
    )

    result = {
        "customer_unique_id": str(customer_unique_id),
        "predicted_future_revenue": (
            float(df.iloc[row_index]["predicted_future_revenue"])
            if "predicted_future_revenue" in df.columns
            else None
        ),
        "top_positive_factors": positive.to_dict(
            orient="records"
        ),
        "top_negative_factors": negative.to_dict(
            orient="records"
        ),
    }

    return result

## 10. Explain the highest predicted-value customer

In [ ]:
highest_value_customer = (
    df.sort_values(
        "predicted_future_revenue",
        ascending=False,
    )
    .iloc[0]["customer_unique_id"]
)

customer_explanation = explain_customer(
    str(highest_value_customer),
    top_n=5,
)

print(
    "Customer:",
    customer_explanation["customer_unique_id"],
)

print(
    "Predicted future revenue:",
    round(
        customer_explanation["predicted_future_revenue"],
        2,
    ),
)

print("\nTop positive factors:")
display(
    pd.DataFrame(
        customer_explanation["top_positive_factors"]
    )
)

print("\nTop negative factors:")
display(
    pd.DataFrame(
        customer_explanation["top_negative_factors"]
    )
)

## 11. Explain a high-value but inactive customer

This identifies customers with:

- high predicted future value,
- relatively high historical recency.

This is particularly useful for retention recommendations.

In [ ]:
inactive_candidates = df.copy()

recency_threshold = inactive_candidates[
    "historical_recency_days"
].quantile(0.75)

value_threshold = inactive_candidates[
    "predicted_future_revenue"
].quantile(0.75)

inactive_candidates = inactive_candidates[
    (inactive_candidates["historical_recency_days"] >= recency_threshold)
    & (
        inactive_candidates["predicted_future_revenue"]
        >= value_threshold
    )
]

if not inactive_candidates.empty:
    candidate = inactive_candidates.sort_values(
        "predicted_future_revenue",
        ascending=False,
    ).iloc[0]

    explanation = explain_customer(
        str(candidate["customer_unique_id"]),
        top_n=5,
    )

    print(
        "High-value inactive customer:",
        explanation["customer_unique_id"],
    )

    print(
        "Predicted future revenue:",
        round(
            explanation["predicted_future_revenue"],
            2,
        ),
    )

    print("\nPositive factors:")
    display(
        pd.DataFrame(
            explanation["top_positive_factors"]
        )
    )

    print("\nNegative factors:")
    display(
        pd.DataFrame(
            explanation["top_negative_factors"]
        )
    )
else:
    print(
        "No high-value inactive customer matched "
        "the current percentile thresholds."
    )

## 12. Save SHAP artifacts

The customer-level SHAP table is saved for downstream APIs and dashboards.

The global feature-importance table is also saved for reporting.

In [ ]:
GLOBAL_SHAP_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "clv_shap_global_importance.csv"
)

shap_df.to_csv(
    SHAP_OUTPUT,
    index=False,
)

global_importance.to_csv(
    GLOBAL_SHAP_PATH,
    index=False,
)

print(f"Saved customer SHAP values: {SHAP_OUTPUT}")
print(f"Saved global SHAP importance: {GLOBAL_SHAP_PATH}")

# Final validation

Expected artifacts:

```text
data/processed/
├── customer_clv_predictions.csv
├── customer_shap_values.csv
└── clv_shap_global_importance.csv
```

The important architectural outcome is:

```text
XGBoost prediction
       +
SHAP explanation
       ↓
Business-readable customer insight
```

The next layer can turn these explanations into retention recommendations and AI-generated business narratives.